# Cycle 1 — Hyperparameter Tuning

**Project:** Football Predictor  
**Depends on:** `cycle1_modelling.ipynb`

---

## Purpose of this Notebook

The baseline modelling notebook trained all models with **default settings**. This notebook finds the **optimal settings** for each model using **Randomized Search Cross Validation**.


## What is Hyperparameter Tuning?

Every ML model has **hyperparameters** — settings you configure before training. For example, XGBoost has:
- `n_estimators` — how many trees to build
- `max_depth` — how deep each tree grows
- `learning_rate` — how fast the model learns
- `subsample` — fraction of data used per tree

Different combinations give different results. Tuning finds the combination that gives the highest accuracy.

## Why Randomized Search over Grid Search?

**Grid Search** tries every possible combination — if you have 5 parameters with 4 options each, that is 4⁵ = 1,024 combinations. Very slow.

**Randomized Search** randomly samples a fixed number of combinations (we use 50). Much faster, and research shows it finds equally good results in practice.

## What is Cross Validation (CV)?

Instead of using one train/test split, CV splits the training data into 5 equal parts (folds). The model trains on 4 folds and tests on the 5th — repeated 5 times. The average score across all 5 folds is the CV score. This gives a more reliable estimate of real performance than a single split.

In [1]:
import sys, os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing



df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))
X1 = df1.drop(columns=['FTR', 'Season'])
y1 = df1['FTR']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {len(X1_train)} | Test: {len(X1_test)} | Features: {X1_train.shape[1]}')


Train: 5472 | Test: 1368 | Features: 33


### Observations
- Same splits as the modelling notebook (same `random_state=42`) — results are directly comparable
- `StratifiedKFold` ensures each fold has the same class distribution as the full dataset — important for imbalanced data

# Tuning XGBoost

## Define Parameter Grid

**What it does:** Defines the range of hyperparameter values to search over.

**Why these parameters?**
- `n_estimators` — more trees = more learning capacity, but slower and risks overfitting
- `max_depth` — deeper trees capture more complexity, but overfit more easily
- `learning_rate` — smaller = more conservative learning, needs more trees to compensate
- `subsample` — fraction of training data used per tree, adding randomness reduces overfitting
- `colsample_bytree` — fraction of features used per tree, reduces correlation between trees
- `min_child_weight` — minimum data points in a leaf, higher = more conservative
- `gamma` — minimum loss reduction for a split, higher = more conservative

In [3]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2]
}

total_combinations = 4 * 4 * 4 * 3 * 3 * 3 * 3
print(f'Total possible combinations: {total_combinations:,}')
print(f'Combinations we will try: 50 (RandomizedSearch)')
print(f'With 5-fold CV: 50 × 5 = 250 model fits')

Total possible combinations: 5,184
Combinations we will try: 50 (RandomizedSearch)
With 5-fold CV: 50 × 5 = 250 model fits


### Observations
- Grid Search would try all 3,888 combinations × 5 folds = 19,440 fits — very slow
- RandomizedSearch tries 250 fits — much faster, finds comparably good results

## Run Randomized Search

**What it does:** Runs 50 random combinations of hyperparameters, each evaluated with 5-fold CV. Returns the best combination.

**Why:** Finds a significantly better XGBoost configuration than the default settings.

In [4]:
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)

search_d1 = RandomizedSearchCV(
    xgb, xgb_param_grid,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_d1.fit(X1_train, y1_train)

print('Best hyperparameters:')
for param, value in search_d1.best_params_.items():
    print(f'  {param}: {value}')
print()
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')

y_pred_xgb_d1_tuned = search_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  subsample: 0.7
  n_estimators: 200
  min_child_weight: 3
  max_depth: 5
  learning_rate: 0.01
  gamma: 0.1
  colsample_bytree: 0.8

Best CV accuracy: 52.47%
Test accuracy:    52.78%


### Observations
- **Low learning rate (0.01) + more trees (300)** — the tuner found that XGBoost learns better slowly on this dataset
- **subsample: 0.7** — using only 70% of data per tree reduces overfitting
- Test accuracy (52.78%) is slightly above CV accuracy (52.47%) — the model generalises well

### Improvement over baseline
- Untuned XGBoost Dataset 1: **50.95%**
- Tuned XGBoost Dataset 1: **52.78%**
- **Gain: +2.49 percentage points**

## Full Classification Report

In [5]:
print('TUNED XGBOOST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_xgb_d1_tuned, target_names=['Away Win', 'Draw', 'Home Win']))

TUNED XGBOOST — Dataset 1
Accuracy: 52.78%

              precision    recall  f1-score   support

    Away Win       0.50      0.39      0.44       394
        Draw       0.42      0.05      0.09       340
    Home Win       0.54      0.87      0.67       634

    accuracy                           0.53      1368
   macro avg       0.49      0.44      0.40      1368
weighted avg       0.50      0.53      0.46      1368



### Observations
- Draw recall remains low (0.06) — even tuned XGBoost struggles with draws on Dataset 1
- Home Win recall is high (0.87) — model confidently identifies home wins
- The season-level form features in Dataset 1 simply do not provide enough signal for draws

---
## Save the tuned Dataset 1 model

The tuned XGBoost on Dataset 1 (Premier League) is the deployable Cycle 1 model. Save the model, scaler, and feature column list for the API.


In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler

# Refit a clean scaler on the Dataset 1 training set so the saved scaler
# matches what the API will see (raw features → scaled → model.predict)
scaler1 = StandardScaler().fit(X1_train)

best_xgb = search_d1.best_estimator_

joblib.dump(best_xgb,          str(Paths.C1_MODEL))
joblib.dump(scaler1,           str(Paths.C1_SCALER))
joblib.dump(list(X1_train.columns), str(Paths.C1_FEATURES))

print(f'Model saved    -> {Paths.C1_MODEL}')
print(f'Scaler saved   -> {Paths.C1_SCALER}')
print(f'Features saved -> {Paths.C1_FEATURES}')
print(f'\nFeature columns ({len(X1_train.columns)}): {list(X1_train.columns)}')

## Key Conclusions

### 1. Tuned XGBoost on Dataset 1 reaches **52.78%** accuracy under random-split CV
Up from the untuned ~50% — a modest but consistent gain. The tuned model is saved for API consumption.

### 2. Random splitting may overstate generalisation
This notebook reports the random-split number for completeness. The honest deployed metric comes from the chronological tuning notebook (`chronological/cycle1_tuning_chronological.ipynb`), which trains on early seasons and tests on later ones.

### 3. Draws remain unsolved
Even the tuned model has near-zero recall on the Draw class. The 33-feature PL set captures form and momentum well but not the situational randomness that produces draws.
